In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv('titanic/train.csv',usecols=['Age','Fare','Survived'])

In [3]:
df.dropna(inplace=True)

In [4]:
df.shape

(714, 3)

In [5]:
df.head()

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [6]:
X = df.drop('Survived',axis=1)  
y = df['Survived']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [8]:
X_train.head()

,Age,Fare
328,31.0,20.5250
73,26.0,14.4542
253,30.0,16.1000
719,33.0,7.7750
666,25.0,13.0000


In [9]:
ct = DecisionTreeClassifier()


In [10]:
ct.fit(X_train,y_train)
y_pred = ct.predict(X_test)

In [11]:
accuracy_score(y_test,y_pred)

0.6293706293706294

In [12]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=5,scoring='accuracy'))

np.float64(0.6260514133753571)

In [13]:
kbin_age = KBinsDiscretizer(n_bins=5,encode='ordinal',strategy='kmeans')
kbin_fare = KBinsDiscretizer(n_bins=5,encode='ordinal',strategy='kmeans')

In [14]:
cf = ColumnTransformer([
    ('kbin_age',kbin_age,[0]),
    ('kbin_fare',kbin_fare,[1])
],remainder='passthrough')

In [15]:
X_train_cf = cf.fit_transform(X_train)
X_test_cf = cf.transform(X_test)

In [16]:
cf.named_transformers_['kbin_age'].n_bins_

array([5])

In [ ]:
cf.named_transformers_['kbin_fare'].bin_edges_

array([array([  0.        ,  42.42381445, 100.62403884, 186.50018901,
              376.60581786, 512.3292    ])                           ],
      dtype=object)

In [18]:
cf.named_transformers_['kbin_age'].bin_edges_

array([array([ 0.42      , 12.69636862, 27.02765256, 39.35418895, 54.21464646,
              80.        ])                                                   ],
      dtype=object)

In [21]:
op = pd.DataFrame({
    'age':X_train['Age'],
    'age_cf':X_train_cf[:,0],
    'fare':X_train['Fare'],
    'fare_cf':X_train_cf[:,1]
})

In [22]:
op

,age,age_cf,fare,fare_cf
328,31.0,2.0,20.5250,0.0
73,26.0,1.0,14.4542,0.0
253,30.0,2.0,16.1000,0.0
719,33.0,2.0,7.7750,0.0
666,25.0,1.0,13.0000,0.0
...,...,...,...,...
92,46.0,3.0,61.1750,1.0
134,25.0,1.0,13.0000,0.0
337,41.0,3.0,134.5000,2.0
548,33.0,2.0,20.5250,0.0


In [23]:
op['age_lb'] = pd.cut(op['age'],bins=cf.named_transformers_['kbin_age'].bin_edges_[0].tolist())
op['fare_lb'] = pd.cut(op['fare'],bins=cf.named_transformers_['kbin_fare'].bin_edges_[0].tolist())

In [24]:
op.head()

,age,age_cf,fare,fare_cf,age_lb,fare_lb
328,31.0,2.0,20.5250,0.0,"(27.028, 39.354]","(0.0, 42.424]"
73,26.0,1.0,14.4542,0.0,"(12.696, 27.028]","(0.0, 42.424]"
253,30.0,2.0,16.1000,0.0,"(27.028, 39.354]","(0.0, 42.424]"
719,33.0,2.0,7.7750,0.0,"(27.028, 39.354]","(0.0, 42.424]"
666,25.0,1.0,13.0000,0.0,"(12.696, 27.028]","(0.0, 42.424]"


In [25]:
clf = DecisionTreeClassifier()
clf.fit(X_train_cf,y_train)
y_pred2 = clf.predict(X_test_cf)

In [26]:
accuracy_score(y_test,y_pred2)

0.6223776223776224